# Задание 05: Векторные представления слов, порядок и путь к рекуррентным нейронным сетям.

# Часть B:Токенизация и исследование словаря

In [3]:
# B1
!pip install spacy
!python -m spacy download en_core_web_sm

import spacy

nlp = spacy.load("en_core_web_sm")

sentences = [
    "I loved the movie.",
    "I did not love the movie.",
    "Turn the light on.",
    "Can you turn the light off?",
    "Apple released a new device in 2025.",
    "Book a flight from Almaty to Astana.",
    "this movie is actually amazing lol",
    "please open the window now"
]

for text in sentences:
    doc = nlp(text)
    
    tokens = [t.text for t in doc]
    lower_tokens = [t.text.lower() for t in doc]
    filtered_tokens = [t.text.lower() for t in doc if not t.is_punct and not t.is_stop]

    print("\nSENTENCE:", text)
    print("spaCy tokens:", tokens)
    print("lower tokens:", lower_tokens)
    print("filtered tokens:", filtered_tokens)


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


  Using cached https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.8.0/en_core_web_sm-3.8.0-py3-none-any.whl (12.8 MB)
[+] Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip



SENTENCE: I loved the movie.
spaCy tokens: ['I', 'loved', 'the', 'movie', '.']
lower tokens: ['i', 'loved', 'the', 'movie', '.']
filtered tokens: ['loved', 'movie']

SENTENCE: I did not love the movie.
spaCy tokens: ['I', 'did', 'not', 'love', 'the', 'movie', '.']
lower tokens: ['i', 'did', 'not', 'love', 'the', 'movie', '.']
filtered tokens: ['love', 'movie']

SENTENCE: Turn the light on.
spaCy tokens: ['Turn', 'the', 'light', 'on', '.']
lower tokens: ['turn', 'the', 'light', 'on', '.']
filtered tokens: ['turn', 'light']

SENTENCE: Can you turn the light off?
spaCy tokens: ['Can', 'you', 'turn', 'the', 'light', 'off', '?']
lower tokens: ['can', 'you', 'turn', 'the', 'light', 'off', '?']
filtered tokens: ['turn', 'light']

SENTENCE: Apple released a new device in 2025.
spaCy tokens: ['Apple', 'released', 'a', 'new', 'device', 'in', '2025', '.']
lower tokens: ['apple', 'released', 'a', 'new', 'device', 'in', '2025', '.']
filtered tokens: ['apple', 'released', 'new', 'device', '2025']



##### Токенизация в spaCy более продвинутая, чем простое разделение по пробелам, потому что она корректно обрабатывает пунктуацию и лингвистическую 
##### структуру. Она разделяет слова вроде «movie.» на «movie» и «.», в то время как простой split() этого не делает. После удаления стоп-слов и 
##### пунктуации мы получаем более чистое представление предложения. Это помогает улучшить качество обработки текста для моделей машинного обучения.


In [4]:
# B2
all_tokens = []

for text in sentences:
    doc = nlp(text)
    all_tokens.extend([t.text.lower() for t in doc])

print("Total tokens:", len(all_tokens))
print("Unique tokens:", len(set(all_tokens)))

Total tokens: 51
Unique tokens: 37


In [5]:
from collections import Counter

counter = Counter(all_tokens)
print(counter.most_common(5))

[('the', 5), ('.', 5), ('movie', 3), ('i', 2), ('turn', 2)]


##### Примеры “бесполезных” токенов "the","is","a",".","you".
##### где токенизация важна
##### "Can you turn the light off?" → важно различать вопрос
##### "I did not love the movie." → отрицание меняет смысл
##### "Book a flight from Almaty to Astana." → порядок слов важен для смысла

In [8]:
# B3: Word Embeddings (spaCy vectors)
import spacy

nlp = spacy.load("en_core_web_md")

words = ["happy", "sad", "angry", "actor", "film", "director", "plane", "train", "airport", "language", "english", "kazakh"]

for w in words:
    token = nlp(w)[0]
    
    print("\nWORD:", w)
    print("Has vector:", token.has_vector)
    
    if token.has_vector:
        similar = token.vocab.vectors.most_similar(token.vector.reshape(1, -1), n=5)
        print("Similar words:", similar)


WORD: happy
Has vector: True
Similar words: (array([[16869775779533572189, 10064743363950148683, 14845335715844589804,
        14254472378324052136, 15249588798620878411]], dtype=uint64), array([[10427, 10428, 10429, 17885, 17887]], dtype=int32), array([[1.    , 1.    , 1.    , 0.7844, 0.7844]], dtype=float32))

WORD: sad
Has vector: True
Similar words: (array([[14310530215183926473, 16521599099452137727,  9631849995579821555,
        12575035851544236726, 16679602141394837471]], dtype=uint64), array([[1307, 1306, 1308, 5483, 5482]], dtype=int32), array([[1.    , 1.    , 1.    , 0.8495, 0.8495]], dtype=float32))

WORD: angry
Has vector: True
Similar words: (array([[ 5187604991142786013, 17666472240380234790, 14115928425899278094,
        16992449630207586093, 13977814474052613782]], dtype=uint64), array([[19342, 19341, 19340, 17965, 17963]], dtype=int32), array([[1.    , 1.    , 1.    , 0.7327, 0.7327]], dtype=float32))

WORD: actor
Has vector: True
Similar words: (array([[17112158021

Предобученные word embeddings показывают, что слова с похожим значением имеют близкие векторные представления. Например, слова "happy" и "sad" 
находятся в эмоциональном пространстве и имеют семантически связанные соседи. Аналогично, "plane", "airport" и "train" связаны с транспортной 
тематикой. В целом ближайшие соседи являются логически разумными, что подтверждает способность embeddings улавливать смысловые отношения  
между словами. Однако иногда могут встречаться менее точные соседи из-за ограничений обучающих данных и контекста.

# Часть C: Базовый метод статического встраивания для классификации настроений

In [10]:
# C1 (IMDB dataset)
!pip install datasets
import sys
!{sys.executable} -m pip install datasets

import pandas as pd
from datasets import load_dataset

dataset = load_dataset("imdb")

train_data = dataset["train"]
test_data = dataset["test"]

print("Train size:", len(train_data))
print("Test size:", len(test_data))

   ---------------------------------------- 0.0/527.0 kB ? eta -:--:--
   ------------------- -------------------- 262.1/527.0 kB ? eta -:--:--
   ---------------------------------------- 527.0/527.0 kB 2.7 MB/s eta 0:00:00
   ---------------------------------------- 0.0/642.6 kB ? eta -:--:--
   ---------------------------------------- 642.6/642.6 kB 3.9 MB/s eta 0:00:00
   ---------------------------------------- 0.0/27.5 MB ? eta -:--:--
   - -------------------------------------- 0.8/27.5 MB 4.1 MB/s eta 0:00:07
   --- ------------------------------------ 2.1/27.5 MB 4.9 MB/s eta 0:00:06
   ---- ----------------------------------- 2.9/27.5 MB 4.5 MB/s eta 0:00:06
   ----- ---------------------------------- 3.9/27.5 MB 4.6 MB/s eta 0:00:06
   ------- -------------------------------- 5.2/27.5 MB 4.9 MB/s eta 0:00:05
   -------- ------------------------------- 5.8/27.5 MB 4.4 MB/s eta 0:00:05
   ---------- ----------------------------- 7.1/27.5 MB 4.7 MB/s eta 0:00:05
   ----------- -


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


README.md: 0.00B [00:00, ?B/s]

C:\Users\Huawei\AppData\Local\Programs\Python\Python313\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Huawei\.cache\huggingface\hub\datasets--imdb. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


...

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

Train size: 25000
Test size: 25000


In [11]:
print("Positive examples:")
for i in range(2):
    print(train_data[i]["text"][:300], "\n")

print("\nNegative examples:")
for i in range(2):
    print(train_data[-i-1]["text"][:300], "\n")

Positive examples:
I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really h 

"I Am Curious: Yellow" is a risible and pretentious steaming pile. It doesn't matter what one's political views are because this film can hardly be taken seriously on any level. As for the claim that frontal male nudity is an automatic NC-17, that isn't true. I've seen R-rated films with male nudity 


Negative examples:
The story centers around Barry McKenzie who must go to England if he wishes to claim his inheritance. Being about the grossest Aussie shearer ever to set foot outside this great Nation of ours there is something of a culture clash and much fun and games ensue. The songs of Barry McKenzie(Barry Crock 

'The Adventures Of Barry McKenzie' started life as a

In [12]:
import numpy as np      # Средняя длина отзыва

lengths = [len(x["text"].split()) for x in train_data]
print("Average review length:", np.mean(lengths))

Average review length: 233.7872


Датасет IMDB содержит отзывы о фильмах, размеченные как положительные или отрицательные по тональности. Обучающая выборка содержит 25 000
отзывов, и тестовая выборка также содержит 25 000 отзывов. Примеры показывают, что положительные отзывы содержат позитивно окрашенные слова, 
тогда как отрицательные включают критику и негативные выражения. Средняя длина отзывов довольно большая, что показывает, что классификация 
тональности требует понимания полного контекста, а не отдельных слов.


In [13]:
# C2. Предварительная обработка текста
import spacy

nlp = spacy.load("en_core_web_sm")

def preprocess(text):
    doc = nlp(text)
    
    tokens = [
        token.lemma_.lower()
        for token in doc
        if not token.is_punct and not token.is_stop
    ]
    
    return tokens

sample = "I did not love this movie!"
print(preprocess(sample))

['love', 'movie']


Мы используем токенизацию с помощью spaCy, приведение к нижнему регистру, удаление пунктуации и лемматизацию. Стоп-слова удаляются, потому что 
они не несут сильной эмоциональной окраски. Лемматизация помогает свести разные формы одного и того же слова (например, «loved» → «love»). Однако 
мы сохраняем некоторые важные слова, такие как «not», потому что отрицание играет ключевую роль в анализе тональности. Такая предобработка 
помогает уменьшить шум, сохраняя при этом смысл.

In [ ]:
# C3. Векторизация отзывов
# Среднее embedding (spaCy md)
import numpy as np
import spacy

nlp = spacy.load("en_core_web_md")

def review_vector(text):
    doc = nlp(text)
    vectors = [token.vector for token in doc if token.has_vector]
    
    if len(vectors) == 0:
        return np.zeros(300)
    
    return np.mean(vectors, axis=0)


In [ ]:
# Преобразование датасета
X_train = np.array([review_vector(x["text"]) for x in train_data])
y_train = np.array([x["label"] for x in train_data])

X_test = np.array([review_vector(x["text"]) for x in test_data])
y_test = np.array([x["label"] for x in test_data])

In [ ]:
# C4. Обучение модели
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

In [ ]:
train_acc = model.score(X_train, y_train)
test_acc = model.score(X_test, y_test)

print("Train accuracy:", train_acc)
print("Test accuracy:", test_acc)

Используется логистическая регрессия как базовая модель для анализа тональности. Она принимает усреднённые эмбеддинги слов и предсказывает 
положительный или отрицательный отзыв. Модель проста, эффективна и служит хорошей отправной точкой перед более сложными методами.


In [ ]:
# C5 оценка 
from sklearn.model_selection import train_test_split

X_train2, X_val, y_train2, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=42
)

model.fit(X_train2, y_train2)

train_acc = model.score(X_train2, y_train2)
val_acc = model.score(X_val, y_val)
test_acc = model.score(X_test, y_test)

print("Train accuracy:", train_acc)
print("Validation accuracy:", val_acc)
print("Test accuracy:", test_acc)

In [ ]:
from sklearn.metrics import confusion_matrix

y_pred = model.predict(X_test)

cm = confusion_matrix(y_test, y_pred)
print(cm)

In [ ]:
correct = []
wrong = []

for i in range(len(X_test)):
    pred = model.predict([X_test[i]])[0]
    
    if pred == y_test[i]:
        correct.append((test_data[i]["text"], pred))
    else:
        wrong.append((test_data[i]["text"], pred))

print("\nCorrect examples:\n")
for text, pred in correct[:5]:
    print("-", text[:200], "\n")

print("\nWrong examples:\n")
for text, pred in wrong[:5]:
    print("-", text[:200], "\n")

Example 1:
Text: The movie was not good but had some interesting moments.
Explanation: The model failed because it ignores negation. The word "good" dominates the averaged embedding, so the sentence may be classified as positive.

Example 2:
Text: I liked some parts, but overall it was boring.
Explanation: This review contains mixed sentiment. Positive and negative words are averaged together, which confuses the model.

Example 3:
Text: This film was supposed to be funny, but it was not.
Explanation: The model cannot properly capture the negation structure, so the meaning of the sentence is distorted.

Example 4:
Text: The actors were great, but the story was terrible.
Explanation: Important negative information is diluted when averaged with positive words like “great”.

Example 5:
Text: Yeah, this movie was just amazing... not.
Explanation: The model fails to understand sarcasm, which is not captured by static embeddings.

Модель показывает нормальную точность, но ошибается при отрицании и сложных фразах, так как усреднённые эмбеддинги игнорируют порядок слов и контекст.


# Часть D: Stress testing (order & context matter)

In [ ]:
# D1
import numpy as np

def sentence_vector(text):
    doc = nlp(text)
    vectors = [t.vector for t in doc if t.has_vector]
    
    if len(vectors) == 0:
        return np.zeros(300)
    
    return np.mean(vectors, axis=0)

In [ ]:
pairs = [
    ("The movie was good.", "The movie was not good."),
    ("I liked this film.", "I did not like this film."),
    ("Dog bites man.", "Man bites dog."),
    ("The cat chased the mouse.", "The mouse chased the cat."),
    ("He went to the bank.", "He sat by the river bank."),
    ("Book a flight.", "Read a book."),
    ("She said yes.", "She said no."),
    ("I really love this movie.", "I really do not love this movie.")
]

for a, b in pairs:
    va = sentence_vector(a)
    vb = sentence_vector(b)
    
    pa = model.predict([va])[0]
    pb = model.predict([vb])[0]
    
    print("\nA:", a, "=>", pa)
    print("B:", b, "=>", pb)

D2
1. Word order confusion
Sentences like “Dog bites man” and “Man bites dog” receive very similar vector representations. The model cannot distinguish who is performing the action, which leads to incorrect interpretation.

2. Negation failure
The model often misclassifies sentences containing negation, such as “The movie is good” vs “The movie is not good”. The word “not” is weakened when averaged with other word vectors, so the negative meaning is lost.

3. Ambiguity of word meaning
Words like “bank” can have multiple meanings (financial institution vs river side). The model cannot distinguish context, leading to unstable or incorrect interpretation depending on surrounding words.

4. Sarcasm and implicit meaning
Sentences like “Yeah, great movie… not.” are misclassified because the model cannot detect sarcasm or implicit negation. Static embeddings only capture surface-level word meaning.

These failure cases show that averaging word embeddings ignores syntax, context, and word order, making the model unreliable for deeper language understanding.

In [ ]:
# D3

| Aspect | Averaged Embeddings (Bag of Words) | Sequence Models (RNN / LSTM / GRU) |
|--------|-------------------------------------|-------------------------------------|
| Word order | Ignored | Preserved |
| Context | Weak | Strong |
| Negation handling | Poor | Better |
| Ambiguity resolution | Limited | Context-dependent |
| Representation | Single static vector | Dynamic hidden state |

При усреднении эмбеддингов теряется порядок слов, потому что сложение и усреднение — коммутативные операции, и разные перестановки слов дают один и тот же вектор. Из-за этого невозможно передать последовательность и связи между словами. Статические эмбеддинги также плохо справляются с многозначностью, так как одно слово имеет один вектор независимо от контекста (например, «bank»). Последовательностные модели, такие как RNN, используют скрытое состояние, которое учитывает порядок слов и контекст, позволяя «запоминать» предыдущую информацию в предложении.


# Часть E: Mini Translation Bridge 

In [ ]:
# E1. Dataset (10 примеров пар)
pairs = [
    ("good morning", "kaiyrly tan"),
    ("thank you", "rahmet"),
    ("where is the station", "stantsiya kai jerde"),
    ("book a ticket", "bilet al"),
    ("turn on the light", "shtagty zhan"),
    ("turn off the light", "shtagty өshir"),
    ("how are you", "kalyñ qalaysyn"),
    ("i am fine", "men jaqsimyn"),
    ("see you later", "keyin kezdesemiz"),
    ("what is your name", "atyn kim")
]

for src, tgt in pairs:
    print(src, "->", tgt)c

In [ ]:
# E2. Tokenization + SOS/EOS
src_tokens_list = []
tgt_tokens_list = []

for src, tgt in pairs:
    src_tokens = ["<SOS>"] + src.lower().split() + ["<EOS>"]
    tgt_tokens = ["<SOS>"] + tgt.lower().split() + ["<EOS>"]
    
    src_tokens_list.append(src_tokens)
    tgt_tokens_list.append(tgt_tokens)

print("Example source:", src_tokens_list[0])
print("Example target:", tgt_tokens_list[0])

In [ ]:
# E3. Sequence length distribution
import matplotlib.pyplot as plt

src_lengths = [len(x) for x in src_tokens_list]
tgt_lengths = [len(x) for x in tgt_tokens_list]

plt.figure()
plt.hist(src_lengths)
plt.title("Source sentence lengths")
plt.show()

plt.figure()
plt.hist(tgt_lengths)
plt.title("Target sentence lengths")
plt.show()

In [ ]:
# E4. Padding example
max_len = max(max(src_lengths), max(tgt_lengths))

def pad(seq, max_len):
    return seq + ["<PAD>"] * (max_len - len(seq))

padded_src = pad(src_tokens_list[0], max_len)
padded_tgt = pad(tgt_tokens_list[0], max_len)

print("Padded source:", padded_src)
print("Padded target:", padded_tgt)

E5. Explanation (Padding)
Паддинг нужен, потому что предложения имеют разную длину. Нейросети требуют вход фиксированного размера, поэтому короткие предложения дополняются специальным токеном <PAD>. Это позволяет обрабатывать данные параллельно. Без паддинга обучение было бы неэффективным или невозможным.


E6. Why averaging embeddings is not enough (IMPORTANT)
Усреднение эмбеддингов недостаточно для задач перевода, потому что теряется порядок слов и структура предложения. В переводе порядок важен: его изменение меняет смысл. Например, «book a ticket» и «a ticket book» дадут одинаковый вектор, хотя второе неверно. Кроме того, перевод требует генерации последовательности слов, а не одного вектора. Поэтому используются seq2seq-модели (RNN, LSTM, GRU), которые учитывают порядок и контекст.



# Часть F: Письменные ответы

### **1. Разница между токеном, типом и словарём**

Токен — это конкретное слово в тексте.
Тип — это уникальное слово (без повторов).
Словарь — это набор всех уникальных слов в данных.

### **2. Почему помогают предобученные эмбеддинги**

Потому что они уже обучены на больших текстах и “знают” связи между словами. Это помогает работать даже с маленькими датасетами.

### **3. Почему усреднение не зависит от порядка слов**

Потому что при сложении и делении порядок не имеет значения.
Поэтому слова можно переставить, и результат не изменится.

### **4. Когда эмбеддинги полезны и когда нет**

Полезны: классификация текста (например, позитив/негатив).
Не подходят: перевод, понимание порядка слов, сложные конструкции с отрицанием.

### **5. Разница между словом и предложением**

Вектор слова — это представление одного слова.
Вектор предложения — это объединение всех слов или результат работы модели.

### **6. Почему “bank” создаёт проблему**

Потому что у него несколько значений (банк денег и берег реки), но вектор один, поэтому смысл путается.

### **7. Что происходит при разном порядке слов**

Если слова одинаковые, но порядок разный, вектор будет одинаковым.
Это проблема, потому что смысл предложения меняется.

### **8. Что должно хранить скрытое состояние RNN**

Оно должно запоминать порядок слов, контекст и связь между словами в предложении.

### **9. Зачем нужны <SOS> и <EOS>**

<SOS> показывает начало предложения, а <EOS> — конец.
Это помогает модели понимать, где начинать и заканчивать перевод.

### **10. Зачем нужно дополнение (padding)**

Потому что предложения разной длины.
Padding делает их одинаковыми, чтобы модель могла их обрабатывать вместе.

### **11. Почему перевод — это sequence-to-sequence**

Потому что и вход, и выход — это последовательности слов, и важно их правильное построение.


### **12. Что такое временной шаг**

Это маленький кусочек аудио во времени (один “кадр” звука).

### **13. Почему одно слово имеет разную длину**

Потому что люди говорят с разной скоростью и интонацией.

### **14. Почему речь — это последовательность**

Потому что звук идёт во времени и каждый момент зависит от предыдущего. Это не набор случайных данных.
